# NeurIPS 2026 Smart Buildings — Version 27: Smith predictor

**Hypothesis:** Explicitly compensating HVAC actuator delay reduces overcorrection and wasted energy compared with the same physical dispatcher without compensation.

**Mechanism:** For each zone, two copies of a nominal five-minute thermal model are simulated. The delay-free copy receives the last issued command; the delayed copy receives a historical command after two full additional five-minute intervals. Their difference adjusts measured comfort error: **T_virtual = T_measured + clip(T_no_delay − T_delayed, ±0.80 K)**. Both models incorporate the same measurement innovation; they are not independent temperature forecasts. The V7-style HVAC equipment dispatcher is unchanged in the ablation.

**Ablations:** `TransferPolicy.ENABLE_SMITH=False` disables compensation but keeps the identical HVAC dispatcher. Setting `ACTUATOR_DELAY_STEPS=0` tests whether the pure-dead-time assumption drives the mechanism.

**Guards:** Bounded per-zone FIFO, strict action-index validation, temperature normalization, rejection of sensor jumps, large-comfort-violation override, explicit episode reset, no model fitting or external artifacts. The entire policy and official Kaggle gateway run in **one code cell**.

**Limitations:** Two intervals of actuator dead time and the thermal coefficients are uncalibrated priors. Plant dynamics vary by building, operating regime and equipment; real delay may be shorter, longer or negligible. Synthetic tests verify contracts and mechanism, NOT official comfort/energy improvements. The reported Version 7 public score is **0.03947** (lower is better); Version 27 is unscored. Compare official paired episodes across buildings and report reward, severe comfort excursions, energy, carbon and control variation.


In [ ]:
"""Version 27: Smith predictor for delayed smart-building HVAC actuation.

The delay-free model receives the latest commanded VAV/coil/reheat input;
the delayed model receives an older commanded input from a bounded FIFO.
Their difference compensates measured temperature before unchanged V7-style
physical HVAC dispatch. Both model states receive the same measurement
innovation. No control model is fitted on hidden evaluation buildings.

Thermal gains and dead-time are explicit engineering priors, not calibrated.
Runtime O(number of zones + AHUs) per five-minute decision.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Smith-compensated comfort feedback and physics-based HVAC dispatch.

    Set ENABLE_SMITH=False for the identical dispatcher without compensation.
    Set ACTUATOR_DELAY_STEPS=0 to ablate dead time in the nominal model.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    ACTIVE_THRESHOLD_K = 0.18
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU still admits at least 30% outside air.
    # Increasing intake beyond that floor also raises exhaust-fan power.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15

    # Input u[t] first contributes to the observation at t+3
    # (two additional five-minute dead-time intervals).
    ACTUATOR_DELAY_STEPS = 2
    ENABLE_SMITH = True

    # Heuristic per-step zone thermal response, not fitted coefficients.
    LEAK_GAIN = 0.003
    AIR_GAIN = 0.025
    REHEAT_GAIN_K = 0.12
    MAX_CORRECTION_K = 0.80
    MAX_SENSOR_STEP_K = 2.0
    SEVERE_VIOLATION_K = 3.0

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0
        outside_stats = meta.get("observation_normalization_metadata", {}).get(
            "outside_air_temperature_sensor", {})
        self.outdoor_mean = float(outside_stats.get("sample_mean", float("nan")))
        self.outdoor_std = float(outside_stats.get("sample_std", float("nan")))

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": self._continuous_index(
                        actuators.get("supply_air_damper_percentage_command")),
                    "valve": self._continuous_index(
                        actuators.get("reheat_valve_percentage_command")),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """Forget thermal model states, delay queues, and diagnostics."""
        self.smith_states = {}
        self.last_smith_correction_k = {}

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _continuous_index(self, index):
        """Reject invalid/negative action indices instead of NumPy wraparound."""
        try:
            position = int(index)
        except (TypeError, ValueError, OverflowError):
            return None
        return position if 0 <= position < self.n_continuous else None

    def _outside_temperature(self, obs, index):
        """Convert the field-normalized outdoor measurement back to Kelvin."""
        raw = self._value(obs, index)
        if not np.isfinite(raw):
            return float("nan")
        if (np.isfinite(self.outdoor_mean)
                and np.isfinite(self.outdoor_std) and self.outdoor_std > 0.0):
            kelvin = raw * self.outdoor_std + self.outdoor_mean
        else:
            kelvin = raw
        return kelvin if 210.0 <= kelvin <= 330.0 else float("nan")

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "key": int(zone["temperature"]),
            "actions": zone,
            "temp": temperature,
            "low": float(low),
            "high": float(high),
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
        }

    # ---- Smith predictor: delayed plant versus delay-free thermal models. ----

    def _thermal_change(self, temperature_k, outdoor_k, command):
        """Nominal temperature delta in one five-minute interval."""
        ambient = outdoor_k if np.isfinite(outdoor_k) else temperature_k
        delta = self.LEAK_GAIN * (ambient - temperature_k)
        if command is not None:
            if command["supply"] is not None:
                delta += (self.AIR_GAIN * command["damper"]
                          * (command["supply"] - temperature_k))
            delta += self.REHEAT_GAIN_K * command["valve"]
        return delta

    def _advance_smith(self, status, outdoor_k):
        """Advance both models with historical, not hypothetical, commands.

        At observation t, the instant model uses u[t-1]; the delayed model
        uses u[t-1-D]. Both are corrected by the same temperature innovation.
        """
        if not self.ENABLE_SMITH:
            return
        key, measured = status["key"], status["temp"]
        if status["actions"]["damper"] is None:
            self.smith_states.pop(key, None)
            return
        state = self.smith_states.get(key)
        if state is None:
            self.smith_states[key] = {
                "instant": measured, "delayed": measured,
                "measured": measured, "outdoor": outdoor_k, "queue": [],
            }
            return  # Cold start: never invent prior pending inputs.

        if abs(measured - state["measured"]) > self.MAX_SENSOR_STEP_K:
            self.smith_states.pop(key, None)
            status["sensor_jump"] = True
            return

        queue = state["queue"]
        immediate = queue[-1] if queue else None
        lag = self.ACTUATOR_DELAY_STEPS + 1
        delayed_input = queue[-lag] if len(queue) >= lag else None
        instant = state["instant"] + self._thermal_change(
            state["instant"], state["outdoor"], immediate)
        delayed = state["delayed"] + self._thermal_change(
            state["delayed"], state["outdoor"], delayed_input)

        # Common innovation prevents the two models from accumulating
        # different unexplained physical disturbances or sensor offsets.
        innovation = measured - delayed
        instant += innovation
        delayed += innovation
        correction = self._clamp(
            instant - delayed, -self.MAX_CORRECTION_K, self.MAX_CORRECTION_K)

        # The delayed state is measurement-aligned. Keep the other model
        # bounded after clipping, even across very long control episodes.
        state["instant"] = measured + correction
        state["delayed"] = measured
        state["measured"] = measured
        state["outdoor"] = outdoor_k

        virtual = measured + correction
        # Never use optimistic model output to underreact to a severe
        # *measured* violation. The old threshold-only floor could turn
        # a 10 K error into a 0.18 K request, dangerously reducing airflow.
        if measured > status["high"] + self.SEVERE_VIOLATION_K:
            virtual = max(virtual, measured)
        elif measured < status["low"] - self.SEVERE_VIOLATION_K:
            virtual = min(virtual, measured)
        # Keep actual temperature for physical air/return mixing; only the
        # hot/cold request is adjusted by pending estimated plant action.
        status["hot"] = max(0.0, virtual - status["high"])
        status["cold"] = max(0.0, status["low"] - virtual)
        self.last_smith_correction_k[key] = correction

    @staticmethod
    def _actuator_fraction(continuous, index):
        """Decode valid [-1,1] actuator command to a 0..1 fraction."""
        if index is None:
            return 0.0
        return float(np.clip((float(continuous[index]) + 1.0) / 2.0, 0.0, 1.0))

    def _remember_commands(self, records, continuous, discrete):
        """Queue the exact issued commands, gated by equipment run states."""
        if not self.ENABLE_SMITH:
            return
        water_on = (0 <= self.water_run < len(discrete)
                    and discrete[self.water_run] == 1)
        active = set()
        for record in records:
            on = record["ahu_on"]
            for status in record["zones"]:
                key, zone = status["key"], status["actions"]
                state = self.smith_states.get(key)
                if state is None or zone["damper"] is None:
                    continue
                if status.get("sensor_jump"):
                    continue
                state["queue"].append({
                    "supply": record["supply_k"] if on else None,
                    "damper": (self._actuator_fraction(continuous, zone["damper"])
                               if on else 0.0),
                    "valve": (self._actuator_fraction(continuous, zone["valve"])
                              if on and water_on else 0.0),
                })
                if len(state["queue"]) > self.ACTUATOR_DELAY_STEPS + 1:
                    state["queue"].pop(0)
                state["outdoor"] = record["outdoor_k"]
                active.add(key)

        # Missing sensors and rejected temperature jumps invalidate memory.
        self.smith_states = {
            key: state for key, state in self.smith_states.items()
            if key in active
        }

    # ---- Shared V7-style physical dispatcher; Smith adjusts comfort error. ----

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _select_air_supply(self, zones, outside_k):
        """Select coil setpoint and air mixing from the plant's heat balance.

        Returns (coil_setpoint_K, return_fraction, delivered_K).

        The cooling coil can only LOWER the mixed-air temperature.
        Economizer mode is enabled only if extra outside air can entirely
        avoid mechanical cooling. Otherwise the guaranteed extra exhaust
        airflow can erase a marginal chiller saving; stay at the 70%
        return-air limit until that trade-off has been measured.
        """
        hottest = max(zone["hot"] for zone in zones)
        return_k = float(np.mean([zone["temp"] for zone in zones]))

        if hottest < self.ACTIVE_THRESHOLD_K:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            hottest_zones = [zone["temp"] for zone in zones
                             if zone["hot"] > 0.0]
            mean_hot_k = float(np.mean(hottest_zones))
            coil_setpoint_k = self._clamp(
                mean_hot_k - 4.0 - 1.3 * hottest,
                self.MIN_SUPPLY_K, 297.0)

        if not np.isfinite(outside_k):
            outside_k = return_k

        return_fraction = self.MAX_RETURN_FRACTION
        if (hottest >= self.ACTIVE_THRESHOLD_K
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            temperature_span = return_k - outside_k
            # The lowest mix reachable at 15% return air must be cool
            # enough to meet the coil setpoint without compressor work.
            min_mix_k = (self.MIN_ECONOMIZER_RECIRC * return_k
                         + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix_k <= coil_setpoint_k:
                required_return = (coil_setpoint_k - outside_k) / temperature_span
                return_fraction = self._clamp(
                    required_return,
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)

        mixed_k = (return_fraction * return_k
                   + (1.0 - return_fraction) * outside_k)
        delivered_k = min(mixed_k, coil_setpoint_k)
        return coil_setpoint_k, return_fraction, delivered_k

    def _control_zone_boxes(self, statuses, delivered, hot_water_demands, continuous):
        """Size VAV flow using estimated heating/cooling temperature headroom.

        In contrast to a fixed damper = gain * temperature-error rule,
        requested damper opening falls as cooling headroom grows. This is
        a *proxy* for delivered flow, not a calibrated CFM calculation.
        Returns (mean_damper, peak_damper) for AHU fan coordination.
        """
        sum_damper = 0.0
        peak_damper = 0.0
        for status in statuses:
            zone = status["actions"]
            hot, cold, temperature = (
                status["hot"], status["cold"], status["temp"])
            if hot >= self.ACTIVE_THRESHOLD_K:
                headroom = max(2.5, temperature - delivered)
                damper = self._clamp(0.12 + 2.1 * hot / headroom,
                                     self.MIN_VAV_FRACTION, 0.98)
            elif cold >= self.ACTIVE_THRESHOLD_K:
                # Heating is VAV reheat, never the AHU cooling coil.
                damper = self._clamp(0.22 + 0.21 * cold, 0.18, 0.88)
                if zone["valve"] is not None:
                    valve = self._clamp(0.16 + 0.31 * cold, 0.0, 1.0)
                    continuous[zone["valve"]] = self._fraction_action(valve)
                    hot_water_demands.append((cold, valve))
            else:
                damper = self.MIN_VAV_FRACTION
            sum_damper += damper
            peak_damper = max(peak_damper, damper)
            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
        # Position fractions are proxies, not calibrated volumetric flow.
        return sum_damper / len(statuses), peak_damper

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Run only if a zone needs conditioning; never call the AHU to heat."""
        if not statuses:
            return
        worst_violation_k = max(max(z["hot"], z["cold"]) for z in statuses)
        if worst_violation_k < self.ACTIVE_THRESHOLD_K:
            return
        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            # Use both typical load and worst VAV. A badly overheated room
            # must not disappear in the average of many neutral zones.
            # The fan model uses airflow ~ sqrt(static pressure).
            demand_proxy = max(mean_damper, 0.50 * peak_damper)
            pressure_pa = self.MIN_FAN_PRESSURE_PA + 1250.0 * np.sqrt(
                self._clamp(
                    (demand_proxy - self.MIN_VAV_FRACTION) / 0.65,
                    0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)
        return delivered

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + 0.35 * max(0.0, 294.0 - outside) + 5.0 * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    # ---- Official five-minute Kaggle policy entry point. ----

    def get_action(self, obs):
        """Observe -> Smith compensation -> physical dispatch -> queue u[t]."""
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        self.last_smith_correction_k = {}
        hot_water_demands, outdoors, records = [], [], []

        for ahu in self.ahus:
            statuses = [
                self._read_zone(obs, zone, fallback) for zone in ahu["zones"]
            ]
            statuses = [status for status in statuses if status is not None]
            outdoor = self._outside_temperature(obs, ahu["outdoor"])
            if np.isfinite(outdoor):
                outdoors.append(outdoor)

            # Only previously ISSUED commands enter this prediction.
            for status in statuses:
                self._advance_smith(status, outdoor)
            delivered = self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
            records.append({
                "zones": statuses,
                "outdoor_k": outdoor,
                "supply_k": delivered,
                "ahu_on": bool(discrete[ahu["run"]]),
            })

        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        self._remember_commands(records, continuous, discrete)
        return {"continuous_action": continuous, "discrete_action": discrete}


# Official Kaggle entrypoint is in this same executable cell as TransferPolicy.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
